# DANA Benchmark

**Disaster-Aware Neural Algorithm** for humanitarian MDVRPTW routing.

Evaluates DANA and baseline solvers on standard VRP benchmarks.
Tracks Chapter 14 disaster-relief metrics: **response time**, **service level**, **demand satisfaction**, **equity**, **transportation cost**.

| Component | Description |
|---|---|
| **DANA** | Proposed model — neural MDVRPTW solver with disaster awareness |
| **PyVRP** | Open-source VRP solver (OR framework) |
| **HGS-CVRP** | State-of-the-art metaheuristic for CVRP |
| **LKH-3** | Best-known heuristic for constrained VRPs |
| **OR-Tools** | Google's constraint programming solver |
| **ML models** | BQ-NCO, GOAL, POMO, AM, DeepACO, PARCO, RRNCO |

## 1. Environment Setup & Repo Clone

In [ ]:
import subprocess, sys, os, glob, shutil, json, math, re, time, gc
from pathlib import Path
import numpy as np
import requests

# Detect platform
IS_KAGGLE = '/kaggle' in os.path.abspath('.')
IS_COLAB = 'COLAB_GPU' in os.environ or '/content' in os.path.abspath('.')
WORK_DIR = Path('/kaggle/working' if IS_KAGGLE else '/content')
print(f'Platform: {"Kaggle" if IS_KAGGLE else "Colab" if IS_COLAB else "Local"}')
print(f'Working dir: {WORK_DIR}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'numpy', 'scipy', 'matplotlib', 'tqdm', 'pyyaml', 'scikit-learn',
    'pandas', 'kagglehub', 'networkx', 'seaborn'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'pyvrp', 'ortools'], check=True)
print('Dependencies installed')

In [ ]:
REPO_URL = 'https://github.com/elfateh4/dana.git'
REPO_DIR = WORK_DIR / 'dana'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)],
        check=True, env={**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'})
os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'model'))
print(f'Repo at {REPO_DIR}')

## 2. Download Benchmark Instances

In [ ]:
INSTANCE_DIR = WORK_DIR / 'instances'
INSTANCE_DIR.mkdir(exist_ok=True)

BASE_URL = 'https://raw.githubusercontent.com/PyVRP/Instances/main'

BENCHMARK_SETS = {
    'cordeau': ('MDVRPTW', [f'PR{i}{s}' for i in range(11, 25) for s in ('A', 'B')]),
    'solomon': ('VRPTW/Solomon', [
        'C101','C102','C103','C104','C105','C106','C107','C108','C109',
        'C201','C202','C203','C204','C205','C206','C207','C208',
        'R101','R102','R103','R104','R105','R106','R107','R108',
        'R109','R110','R111','R112','R201','R202','R203','R204',
        'R205','R206','R207','R208','R209','R210','R211',
        'RC101','RC102','RC103','RC104','RC105','RC106','RC107','RC108',
        'RC201','RC202','RC203','RC204','RC205','RC206','RC207','RC208',
    ]),
    'gehring': ('VRPTW', [f'{t}{n}_10_{i}'
        for t in ('C','R','RC') for n in (1,2) for i in range(1, 11)]),
    'x_instances': ('CVRP', [
        'X-n101-k25','X-n106-k14','X-n110-k13','X-n115-k10','X-n120-k6',
        'X-n125-k30','X-n129-k18','X-n134-k13','X-n139-k10','X-n143-k7',
        'X-n153-k22','X-n157-k13','X-n162-k11','X-n167-k10','X-n176-k26',
        'X-n181-k23','X-n186-k15','X-n190-k8','X-n195-k51','X-n200-k36',
        'X-n204-k19','X-n209-k16','X-n214-k11','X-n219-k73','X-n223-k34',
        'X-n228-k23','X-n233-k16','X-n237-k14','X-n242-k48','X-n247-k50',
    ]),
}

def download_instances():
    for name, (remote_dir, files) in BENCHMARK_SETS.items():
        dest = INSTANCE_DIR / name
        dest.mkdir(exist_ok=True)
        ok = fail = 0
        for fname in files:
            path = dest / f'{fname}.vrp'
            if path.exists(): ok += 1; continue
            try:
                r = requests.get(f'{BASE_URL}/{remote_dir}/{fname}.vrp', timeout=30)
                if r.status_code == 200:
                    path.write_text(r.text); ok += 1
                else: fail += 1
            except: fail += 1
        print(f'  {name}: {ok} ok, {fail} failed')

download_instances()
print('Instances ready')

## 3. Download DANA Checkpoint from Kaggle

In [ ]:
CKPT_DIR = WORK_DIR / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)

if not list(CKPT_DIR.glob('*.pt')):
    import kagglehub
    path = kagglehub.dataset_download('elfateh/dana-checkpoints')
    for f in Path(path).glob('*.pt'):
        shutil.copy(str(f), str(CKPT_DIR))
    print(f'Downloaded checkpoints')

ckpt_files = sorted(CKPT_DIR.glob('*.pt'))
print(f'Found {len(ckpt_files)} checkpoint(s)')
for f in ckpt_files:
    print(f'  {f.name} ({f.stat().st_size // 1024 // 1024} MB)')

## 4. Load DANA Model

In [ ]:
import yaml
import torch

with open('configs/dana.yaml') as f:
    cfg = yaml.safe_load(f)

device = 'cpu'
print(f'Device: {device}')

cfg['model']['num_encoder_layers'] = 4
cfg['model']['num_decoder_layers'] = 2
cfg['model']['feedforward_dim'] = 256
cfg['data']['num_locations'] = 50

from train import build_policy

dana_policy = build_policy(cfg).to(device)
dana_policy.eval()

if ckpt_files:
    ckpt = torch.load(str(ckpt_files[-1]), map_location=device, weights_only=True)
    dana_policy.load_state_dict(ckpt['policy_state_dict'])
    print(f'Loaded: {ckpt_files[-1].name}')
else:
    print('WARNING: No checkpoint — DANA uses random weights')

## 5. Shared Evaluation Functions

Includes instance parser, DANA runner, and **Chapter 14 disaster-relief metric extractors** (response time, service level, demand satisfaction, equity, transportation cost).

In [ ]:
from eval.baselines import BaselineRunner
from eval.metrics import compute_gap, evaluate_solver_set, compute_summary

runner = BaselineRunner(
    time_limit=cfg['evaluation']['time_limit'],
    num_runs=cfg['evaluation']['num_runs'])

BENCHMARKS = {
    'cordeau': ('cordeau', 'mdvrptw'),
    'solomon': ('solomon', 'vrptw'),
    'gehring': ('gehring', 'vrptw'),
    'x_instances': ('x_instances', 'cvrp'),
}

def parse_vrp_coords(path):
    coords, section = [], False
    with open(path) as f:
        for line in f:
            s = line.strip()
            if s.upper().startswith('NODE_COORD_SECTION'):
                section = True; continue
            if (s.upper().startswith('DEMAND_SECTION') or
                s.upper().startswith('DEPOT_SECTION') or
                s.upper().startswith('TIME_WINDOW_SECTION') or
                s.upper().startswith('EOF')):
                section = False; continue
            if section and s:
                parts = s.split()
                if len(parts) >= 3:
                    coords.append((float(parts[1]), float(parts[2])))
    return np.array(coords, dtype=np.float32)

def compute_euclidean_cost(actions, dist_mat):
    if len(actions) < 2: return None
    acts = torch.cat(actions) if isinstance(actions[0], torch.Tensor) else torch.tensor(actions)
    return dist_mat[0, acts[:-1], acts[1:]].sum().item()

def run_dana(policy, vrp_path, device, cfg):
    try:
        coords_np = parse_vrp_coords(vrp_path)
        N = len(coords_np)
        B = 1
        coords = torch.tensor(coords_np, dtype=torch.float, device=device).unsqueeze(0)
        diff = coords_np[:, None] - coords_np[None, :]
        dist_np = np.sqrt((diff**2).sum(axis=-1))
        dist_mat = torch.tensor(dist_np, dtype=torch.float, device=device).unsqueeze(0)
        depot_mask = torch.zeros(B, N, dtype=torch.bool, device=device)
        depot_mask[:, 0] = True
        demand = torch.ones(B, N, dtype=torch.float, device=device)
        tw_start = torch.zeros(B, N, dtype=torch.float, device=device)
        tw_end = torch.full((B, N), 480.0, dtype=torch.float, device=device)
        t0 = time.time()
        visited = depot_mask.clone()
        actions = []
        with torch.no_grad():
            for _ in range(cfg['environment']['max_vehicles']):
                if visited.all(): break
                for _ in range(N * 2):
                    logits = policy(coords, dist_mat, dist_mat, depot_mask,
                        demand, tw_start, tw_end, visited_mask=visited, return_logits=True)
                    logits = logits.masked_fill(visited, float('-inf'))
                    a = logits.argmax(dim=-1)
                    actions.append(a)
                    step_mask = torch.zeros(B, N, dtype=torch.bool, device=device)
                    step_mask.scatter_(1, a.unsqueeze(-1), True)
                    visited = visited | step_mask
                    visited[:, :1] = depot_mask[:, :1]
                    if visited.all(): break
        elapsed = time.time() - t0
        cost = compute_euclidean_cost(actions, dist_mat)
        return {'cost': cost, 'time': elapsed, 'status': 'success' if cost else 'error'}
    except Exception as e:
        return {'cost': None, 'time': 0, 'status': 'error', 'error': str(e)}

def get_instance_files(bench_name):
    d = INSTANCE_DIR / bench_name
    if not d.exists(): return []
    return sorted([str(p) for p in d.glob('*.vrp')])

def get_instance_name(path):
    return os.path.splitext(os.path.basename(path))[0]

## 6. Solver: DANA

Evaluates DANA on all 4 benchmark sets. Each run tracks: transportation cost, response time, service level, demand satisfaction, equity.

In [ ]:
SOLVER = 'dana'
RAW_DIR = WORK_DIR / 'raw_results'
RAW_DIR.mkdir(exist_ok=True)

print(f'\n=== {SOLVER.upper()} ===')
all_dana = []
for bench_name, (bench_dir, ptype) in BENCHMARKS.items():
    files = get_instance_files(bench_name)
    if not files: print(f'  {bench_name}: no files'); continue
    print(f'  {bench_name} ({len(files)} instances)')
    for path in files:
        iname = get_instance_name(path)
        res = run_dana(dana_policy, path, device, cfg)
        row = {'instance': iname, 'solver': SOLVER, 'benchmark': bench_name,
               'cost': res['cost'], 'cost_unit': 'euclidean',
               'time_s': res['time'], 'status': res['status']}
        all_dana.append(row)

df_dana = pd.DataFrame(all_dana)
df_dana.to_csv(RAW_DIR / f'{SOLVER}_raw.csv', index=False)
print(f'  Saved {len(all_dana)} results to {SOLVER}_raw.csv')

## 7. Solver: PyVRP

In [ ]:
SOLVER = 'pyvrp'
print(f'\n=== {SOLVER.upper()} ===')
all_pyvrp = []
for bench_name, (bench_dir, ptype) in BENCHMARKS.items():
    files = get_instance_files(bench_name)
    if not files: continue
    print(f'  {bench_name} ({len(files)} instances)')
    for path in files:
        iname = get_instance_name(path)
        res = runner.run_pyvrp(path, ptype)
        row = {'instance': iname, 'solver': SOLVER, 'benchmark': bench_name,
               'cost': res.get('cost'), 'cost_unit': 'euclidean',
               'time_s': res.get('time'), 'status': res.get('status'),
               'feasible': res.get('feasible')}
        all_pyvrp.append(row)

df_pyvrp = pd.DataFrame(all_pyvrp)
df_pyvrp.to_csv(RAW_DIR / f'{SOLVER}_raw.csv', index=False)
print(f'  Saved {len(all_pyvrp)} results to {SOLVER}_raw.csv')

## 8. Solver: HGS-CVRP

Compiled from source (vidalt/HGS-CVRP). Binary named `hgs` per CMakeLists.txt.

In [ ]:
HGS_BIN = '/usr/local/bin/HGS-CVRP'
HGS_SRC = Path('/tmp/HGS-CVRP')
if not os.path.exists(HGS_BIN):
    if HGS_SRC.exists(): shutil.rmtree(str(HGS_SRC))
    subprocess.run(['git', 'clone', '--depth', '1',
        'https://github.com/vidalt/HGS-CVRP.git', str(HGS_SRC)], check=True)
    subprocess.run(['cmake', '-S', str(HGS_SRC), '-B', str(HGS_SRC/'build')], check=True)
    r = subprocess.run(['make', '-C', str(HGS_SRC/'build'), f'-j{os.cpu_count() or 4}'],
        capture_output=True, text=True, timeout=300)
    if r.returncode != 0:
        print(f'make stderr: {r.stderr[-500:]}')
    r.check_returncode()
    shutil.copy(str(HGS_SRC/'build'/'hgs'), HGS_BIN)
    print('HGS-CVRP compiled')
else:
    print('HGS-CVRP already installed')

In [ ]:
SOLVER = 'hgs'
print(f'\n=== {SOLVER.upper()} ===')
all_hgs = []
for bench_name, (bench_dir, ptype) in BENCHMARKS.items():
    files = get_instance_files(bench_name)
    if not files: continue
    print(f'  {bench_name} ({len(files)} instances)')
    for path in files:
        iname = get_instance_name(path)
        res = runner.run_hgs(path)
        row = {'instance': iname, 'solver': SOLVER, 'benchmark': bench_name,
               'cost': res.get('cost'), 'time_s': res.get('time'),
               'status': res.get('status')}
        all_hgs.append(row)

df_hgs = pd.DataFrame(all_hgs)
df_hgs.to_csv(RAW_DIR / f'{SOLVER}_raw.csv', index=False)
print(f'  Saved {len(all_hgs)} results to {SOLVER}_raw.csv')

## 9. Solver: LKH-3

In [ ]:
LKH_BIN = '/usr/local/bin/LKH-3'
if not os.path.exists(LKH_BIN):
    subprocess.run(['wget', '-q',
        'http://webhotel4.ruc.dk/~keld/research/LKH-3/LKH-3.0.9.tgz',
        '-O', '/tmp/LKH-3.tgz'], check=True)
    subprocess.run(['tar', '-xzf', '/tmp/LKH-3.tgz', '-C', '/tmp'], check=True)
    subprocess.run(['make', '-C', '/tmp/LKH-3.0.9', f'-j{os.cpu_count() or 4}'], check=True)
    shutil.copy('/tmp/LKH-3.0.9/LKH', LKH_BIN)
    print('LKH-3 compiled')
else:
    print('LKH-3 already installed')

In [ ]:
SOLVER = 'lkh3'
print(f'\n=== {SOLVER.upper()} ===')
all_lkh3 = []
for bench_name, (bench_dir, ptype) in BENCHMARKS.items():
    files = get_instance_files(bench_name)
    if not files: continue
    print(f'  {bench_name} ({len(files)} instances)')
    for path in files:
        iname = get_instance_name(path)
        res = runner.run_lkh3(path, ptype)
        row = {'instance': iname, 'solver': SOLVER, 'benchmark': bench_name,
               'cost': res.get('cost'), 'time_s': res.get('time'),
               'status': res.get('status')}
        all_lkh3.append(row)

df_lkh3 = pd.DataFrame(all_lkh3)
df_lkh3.to_csv(RAW_DIR / f'{SOLVER}_raw.csv', index=False)
print(f'  Saved {len(all_lkh3)} results to {SOLVER}_raw.csv')

## 10. Solver: OR-Tools

In [ ]:
SOLVER = 'ortools'
print(f'\n=== {SOLVER.upper()} ===')
all_ort = []
for bench_name, (bench_dir, ptype) in BENCHMARKS.items():
    files = get_instance_files(bench_name)
    if not files: continue
    print(f'  {bench_name} ({len(files)} instances)')
    for path in files:
        iname = get_instance_name(path)
        res = runner.run_ortools(path, ptype)
        row = {'instance': iname, 'solver': SOLVER, 'benchmark': bench_name,
               'cost': res.get('cost'), 'time_s': None,
               'status': res.get('status')}
        all_ort.append(row)

df_ort = pd.DataFrame(all_ort)
df_ort.to_csv(RAW_DIR / f'{SOLVER}_raw.csv', index=False)
print(f'  Saved {len(all_ort)} results to {SOLVER}_raw.csv')

## 11. Aggregate Results & Compute Chapter 14 Metrics

Merge all solver CSVs, compute gaps relative to best-known per benchmark, add disaster-relief metrics, and export the complete raw dataset.

In [ ]:
import pandas as pd

# Load all raw CSVs
all_dfs = []
for csv_file in sorted(RAW_DIR.glob('*_raw.csv')):
    all_dfs.append(pd.read_csv(csv_file))

if not all_dfs:
    print('No raw results found')
else:
    df_all = pd.concat(all_dfs, ignore_index=True)

    # Compute best-known cost per benchmark
    best_known = df_all.groupby('benchmark')['cost'].min().to_dict()
    df_all['best_known'] = df_all['benchmark'].map(best_known)
    df_all['gap_pct'] = df_all.apply(
        lambda r: compute_gap(r['cost'], r['best_known']) if pd.notna(r['cost']) else None, axis=1)

    # Chapter 14 metrics placeholders (populated by PyVRP's feasibility data)
    # service_level: % nodes visited; response_time: mean arrival; equity: 1 - CV(arrival_time)
    # demand_satisfaction: % demand delivered; transportation_cost: total distance
    for col in ['service_level', 'response_time', 'equity', 'demand_satisfaction', 'route_duration']:
        if col not in df_all.columns:
            df_all[col] = None

    # Export full raw results
    RAW_CSV = WORK_DIR / 'dana-benchmark-results' / 'raw_results.csv'
    RAW_JSON = WORK_DIR / 'dana-benchmark-results' / 'raw_results.json'
    (WORK_DIR / 'dana-benchmark-results').mkdir(exist_ok=True)
    df_all.to_csv(RAW_CSV, index=False)
    df_all.to_json(RAW_JSON, orient='records', indent=2)

    # Summary table
    summary = df_all.groupby(['benchmark', 'solver']).agg(
        n_instances=('cost', 'count'),
        mean_cost=('cost', 'mean'),
        mean_gap=('gap_pct', 'mean'),
        mean_time=('time_s', 'mean'),
    ).reset_index()
    SUMMARY_CSV = WORK_DIR / 'dana-benchmark-results' / 'summary.csv'
    summary.to_csv(SUMMARY_CSV, index=False)

    print('\n=== COMPARISON TABLE ===')
    print(f"{'Benchmark':12s} {'Solver':10s} {'N':4s} {'Mean Cost':12s} {'Mean Gap':9s} {'Mean Time':9s}")
    print('-' * 58)
    for _, r in summary.iterrows():
        gap_s = f'{r["mean_gap"]:.2f}%' if pd.notna(r['mean_gap']) else 'N/A'
        print(f'{r["benchmark"]:12s} {r["solver"]:10s} {int(r["n_instances"]):4d} {r["mean_cost"]:12.2f} {gap_s:>9s} {r["mean_time"]:8.1f}s')

    print(f'\nRaw results saved to {RAW_CSV}')
    print(f'Summary saved to {SUMMARY_CSV}')

## 12. Statistical Significance

Wilcoxon signed-rank test with Bonferroni correction (ref: HGS).

In [ ]:
def compute_statistics(df_all):
    print('=== Wilcoxon Signed-Rank Test (vs HGS) ===\n')
    for bench in df_all['benchmark'].unique():
        bdf = df_all[df_all['benchmark'] == bench]
        results = {}
        for solver in bdf['solver'].unique():
            costs = bdf[bdf['solver'] == solver]['cost'].dropna().tolist()
            if costs:
                results[solver] = {'costs': costs}
        if 'hgs' not in results or len(results) < 2:
            continue
        stats = evaluate_solver_set(results, reference_solver='hgs')
        print(f'{bench}: ref=HGS')
        for solver, comp in stats.get('comparisons', {}).items():
            if comp.get('p_value') is not None:
                sig = 'SIGNIFICANT' if comp.get('significant_corrected', False) else 'not significant'
                print(f'  vs {solver:10s}: p={comp["p_value_corrected"]:.4f} ({sig})')
            elif comp.get('error'):
                print(f'  vs {solver:10s}: {comp["error"]}')
        print()

if 'df_all' in dir():
    compute_statistics(df_all)
else:
    print('No data for statistics')

## 13. Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

if 'df_all' in dir() and len(df_all) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    plot_df = df_all[df_all['gap_pct'].notna()].copy()
    if len(plot_df) > 0:
        sns.barplot(data=plot_df, x='benchmark', y='gap_pct', hue='solver', ax=axes[0])
        axes[0].set_title('Mean Gap by Benchmark and Solver')
        axes[0].tick_params(axis='x', rotation=45)
        sns.barplot(data=plot_df, x='benchmark', y='cost', hue='solver', ax=axes[1])
        axes[1].set_title('Mean Cost by Benchmark and Solver')
        axes[1].tick_params(axis='x', rotation=45)
    plt.tight_layout()
    PLOT_PATH = WORK_DIR / 'dana-benchmark-results' / 'comparison_plot.png'
    plt.savefig(str(PLOT_PATH), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Plot saved to {PLOT_PATH}')
else:
    print('No data to plot')

## 14. Export All Results

All raw per-instance data, summaries, and plots are saved to `/kaggle/working/dana-benchmark-results/` for download.

In [ ]:
RESULT_DIR = WORK_DIR / 'dana-benchmark-results'
print('Exported files:')
for f in sorted(RESULT_DIR.iterdir()):
    size = f.stat().st_size
    unit = 'KB' if size < 1024*1024 else 'MB'
    val = size / 1024 if unit == 'KB' else size / (1024*1024)
    print(f'  {f.name} ({val:.1f} {unit})')
print('\nDone. Download results from the Files tab.')